## Early Fusion 

In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
import json
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_score, recall_score, average_precision_score

import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn.metrics._ranking")

In [2]:
image_features_path = "image_features.npy"    # dict: {img_id: feature}
text_features_val   = "text_features_val.npy" # dict: {img_id: [5,768]}
instances_json      = "/home/BTECH_7TH_SEM/Desktop/MML/MS-COCO/annotations_trainval2017/annotations/instances_val2017.json"

# Load features
image_feats = np.load(image_features_path, allow_pickle=True).item()["features"]
text_feats  = np.load(text_features_val, allow_pickle=True).item()

In [3]:
instances_json = "/home/BTECH_7TH_SEM/Desktop/MML/MS-COCO/annotations_trainval2017/annotations/instances_val2017.json"

def load_coco_labels_complete(json_file, image_feats):
    with open(json_file, "r") as f:
        coco_data = json.load(f)

    # dynamically remap category IDs
    all_cat_ids = sorted({ann["category_id"] for ann in coco_data["annotations"]})
    catid2idx = {cat_id: idx for idx, cat_id in enumerate(all_cat_ids)}
    num_classes = len(all_cat_ids)

    # Build label mapping
    img_to_labels = {}
    for ann in coco_data["annotations"]:
        img_id = ann["image_id"]
        cat_id = ann["category_id"]
        idx = catid2idx[cat_id]
        img_to_labels.setdefault(img_id, set()).add(idx)

    # Ensure every image in image_feats has a label vector
    id_to_label = {}
    for img_id in image_feats.keys():
        multi_hot = np.zeros(num_classes, dtype=np.float32)
        if img_id in img_to_labels:
            for c in img_to_labels[img_id]:
                multi_hot[c] = 1
        # if image has no annotation, multi_hot stays all zeros
        id_to_label[img_id] = multi_hot

    print(f"Detected {num_classes} classes in dataset")
    print(f"Total images with labels: {len(id_to_label)}")
    return id_to_label

# Usage
labels = load_coco_labels_complete(instances_json, image_feats)

Detected 80 classes in dataset
Total images with labels: 5000


In [4]:
combined_dataset = []

for img_id, t_feats in text_feats.items():
    if img_id not in image_feats or img_id not in labels:
        continue
    i_feat = image_feats[img_id]  # (2048,)
    y = labels[img_id]            # (num_classes,)
    
    for t_feat in t_feats:        # per caption
        fused = np.concatenate([i_feat, t_feat], axis=-1)  # (2048 + 768,)
        combined_dataset.append((img_id, fused, y))

print("Total fused samples:", len(combined_dataset))
print("Feature dimension:", combined_dataset[0][1].shape)
print("Label dimension:", combined_dataset[0][2].shape)

Total fused samples: 25014
Feature dimension: (2816,)
Label dimension: (80,)


In [5]:
# saving csv file of combined_dataset
rows = []
for img_id, fused, label in combined_dataset:
    fused_str = ",".join(map(str, fused.tolist()))
    label_str = ",".join(map(str, label.tolist()))
    rows.append([img_id, fused_str, label_str])

df = pd.DataFrame(rows, columns=["image_id", "features", "labels"])
df.to_csv("combined_dataset.csv", index=False)
print("Saved combined dataset to combined_dataset.csv")

Saved combined dataset to combined_dataset.csv


## MLP (Training the Model)

In [6]:
df = pd.read_csv("combined_dataset.csv")

# Parse features and labels from comma-separated strings
df["features"] = df["features"].apply(lambda x: np.array(list(map(float, x.split(","))), dtype=np.float32))
df["labels"]   = df["labels"].apply(lambda x: np.array(list(map(float, x.split(","))), dtype=np.float32))

print("Total samples:", len(df))
print("Feature shape:", df["features"].iloc[0].shape)
print("Label shape:", df["labels"].iloc[0].shape)


Total samples: 25014
Feature shape: (2816,)
Label shape: (80,)


In [7]:
all_img_ids = df["image_id"].unique()

train_ids, test_ids = train_test_split(all_img_ids, test_size=0.2, random_state=42)
train_ids, val_ids  = train_test_split(train_ids, test_size=0.125, random_state=42)  # 0.125*0.8=0.1

train_df = df[df["image_id"].isin(train_ids)]
val_df   = df[df["image_id"].isin(val_ids)]
test_df  = df[df["image_id"].isin(test_ids)]

print("Train samples:", len(train_df))
print("Val samples:", len(val_df))
print("Test samples:", len(test_df))

Train samples: 17510
Val samples: 2502
Test samples: 5002


In [8]:
class FusionDataset(Dataset):
    def __init__(self, df):
        self.features = df["features"].tolist()
        self.labels = df["labels"].tolist()

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        feat = torch.tensor(self.features[idx], dtype=torch.float32)
        label = torch.tensor(self.labels[idx], dtype=torch.float32)
        return feat, label

train_set = FusionDataset(train_df)
val_set   = FusionDataset(val_df)
test_set  = FusionDataset(test_df)

In [9]:
train_loader = DataLoader(train_set, batch_size=128, shuffle=True)
val_loader   = DataLoader(val_set, batch_size=128)
test_loader  = DataLoader(test_set, batch_size=128)

# check one batch
for feats, labels in train_loader:
    print("Batch features shape:", feats.shape)
    print("Batch labels shape:", labels.shape)
    break

Batch features shape: torch.Size([128, 2816])
Batch labels shape: torch.Size([128, 80])


In [10]:
class FusionMLP(nn.Module):
    def __init__(self, input_dim=2816, hidden_dim=2048, num_classes=80):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.5),

            nn.Linear(hidden_dim, hidden_dim // 2),  # 1024
            nn.BatchNorm1d(hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(0.5),

            nn.Linear(hidden_dim // 2, hidden_dim // 4),  # 512
            nn.BatchNorm1d(hidden_dim // 4),
            nn.ReLU(),
            nn.Dropout(0.4),

            nn.Linear(hidden_dim // 4, hidden_dim // 8),  # 256
            nn.BatchNorm1d(hidden_dim // 8),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(hidden_dim // 8, num_classes),
            nn.Sigmoid()  # multi-label classification
        )

    def forward(self, x):
        return self.net(x)

# Get input and output dimensions
input_dim = train_set[0][0].shape[0]   # e.g. 2816
num_classes = train_set[0][1].shape[0] # e.g. 80

# Define model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = FusionMLP(input_dim=input_dim, num_classes=num_classes).to(device)

In [11]:
criterion = nn.BCELoss()  # multi-label classification
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)

In [12]:
def multilabel_accuracy(y_true, y_pred, threshold=0.5):
    """
    Compute per-sample accuracy:
    For each sample, compute (TP+TN)/(Total labels), then average over all samples.
    """
    y_pred_bin = (y_pred >= threshold).astype(int)
    # (N, C)
    correct_per_sample = (y_pred_bin == y_true).sum(axis=1)  
    total_labels = y_true.shape[1]
    sample_acc = correct_per_sample / total_labels
    return sample_acc.mean()

def sample_f1(y_true, y_pred, threshold=0.5):
    y_pred_bin = (y_pred >= threshold).astype(int)
    return f1_score(y_true, y_pred_bin, average="samples")

def evaluate(loader, threshold=0.5):
    """Compute mAP, sample accuracy and sample F1/Precision/Recall."""
    model.eval()
    all_labels, all_preds = [], []
    with torch.no_grad():
        for feats, labels in loader:
            feats, labels = feats.to(device), labels.to(device)
            preds = model(feats).cpu().numpy()
            all_preds.append(preds)
            all_labels.append(labels.cpu().numpy())
    all_preds = np.vstack(all_preds)
    all_labels = np.vstack(all_labels)

    # binarize predictions
    all_preds_bin = (all_preds >= threshold).astype(int)

    # metrics
    mAP = average_precision_score(all_labels, all_preds, average="macro")
    acc_sample = multilabel_accuracy(all_labels, all_preds, threshold)   # << new accuracy
    
    # sample-level F1, precision, recall
    sample_f1 = f1_score(all_labels, all_preds_bin, average="samples", zero_division=0)
    sample_prec = precision_score(all_labels, all_preds_bin, average="samples", zero_division=0)
    sample_rec = recall_score(all_labels, all_preds_bin, average="samples", zero_division=0)

    return mAP, acc_sample, sample_f1, sample_prec, sample_rec

In [13]:
num_epochs = 30
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for feats, labels in train_loader:
        feats, labels = feats.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(feats)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg_train_loss = total_loss / len(train_loader)

    model.eval()
    val_loss = 0
    with torch.no_grad():
        for feats, labels in val_loader:
            feats, labels = feats.to(device), labels.to(device)
            outputs = model(feats)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
    avg_val_loss = val_loss / len(val_loader)

    # Compute validation metrics
    val_mAP, val_acc, val_f1, val_prec, val_rec = evaluate(val_loader)

    print(f"Epoch {epoch+1}/{num_epochs} "
              f"- Train Loss: {avg_train_loss:.4f} "
              f"- Val Loss: {avg_val_loss:.4f} "
              f"- Val mAP: {val_mAP:.4f} "
              f"- Val Sample Acc: {val_acc:.4f} "
              f"- Val F1: {val_f1:.4f} "
              f"- Val Prec: {val_prec:.4f} "
              f"- Val Rec: {val_rec:.4f}")


Epoch 1/30 - Train Loss: 0.4740 - Val Loss: 0.3875 - Val mAP: 0.3691 - Val Sample Acc: 0.9672 - Val F1: 0.3689 - Val Prec: 0.5698 - Val Rec: 0.2978
Epoch 2/30 - Train Loss: 0.2326 - Val Loss: 0.2062 - Val mAP: 0.4570 - Val Sample Acc: 0.9697 - Val F1: 0.3039 - Val Prec: 0.5573 - Val Rec: 0.2223
Epoch 3/30 - Train Loss: 0.1551 - Val Loss: 0.1395 - Val mAP: 0.5028 - Val Sample Acc: 0.9697 - Val F1: 0.3024 - Val Prec: 0.5520 - Val Rec: 0.2208
Epoch 4/30 - Train Loss: 0.1245 - Val Loss: 0.1139 - Val mAP: 0.5277 - Val Sample Acc: 0.9702 - Val F1: 0.3239 - Val Prec: 0.5742 - Val Rec: 0.2416
Epoch 5/30 - Train Loss: 0.1090 - Val Loss: 0.1007 - Val mAP: 0.5413 - Val Sample Acc: 0.9712 - Val F1: 0.3568 - Val Prec: 0.5941 - Val Rec: 0.2761
Epoch 6/30 - Train Loss: 0.0993 - Val Loss: 0.0919 - Val mAP: 0.5540 - Val Sample Acc: 0.9721 - Val F1: 0.3948 - Val Prec: 0.6268 - Val Rec: 0.3174
Epoch 7/30 - Train Loss: 0.0919 - Val Loss: 0.0865 - Val mAP: 0.5656 - Val Sample Acc: 0.9729 - Val F1: 0.4316 -

In [15]:
train_mAP, train_acc, train_f1, train_prec, train_rec = evaluate(train_loader)
val_mAP, val_acc, val_f1, val_prec, val_rec = evaluate(val_loader)
test_mAP, test_acc, test_f1, test_prec, test_rec = evaluate(test_loader)

print("\n=== Final Results of Multimodal Dataset (Image + Text)===")
print(f"Train: mAP={train_mAP:.4f}, Acc={train_acc:.4f}, "
      f"F1={train_f1:.4f}, Prec={train_prec:.4f}, Rec={train_rec:.4f}")
print(f"Val:   mAP={val_mAP:.4f}, Acc={val_acc:.4f}, "
      f"F1={val_f1:.4f}, Prec={val_prec:.4f}, Rec={val_rec:.4f}")
print(f"Test:  mAP={test_mAP:.4f}, Acc={test_acc:.4f}, "
      f"F1={test_f1:.4f}, Prec={test_prec:.4f}, Rec={test_rec:.4f}")



=== Final Results of Multimodal Dataset (Image + Text)===
Train: mAP=0.9367, Acc=0.9938, F1=0.9291, Prec=0.9541, Rec=0.9194
Val:   mAP=0.6484, Acc=0.9774, F1=0.6692, Prec=0.7484, Rec=0.6574
Test:  mAP=0.6545, Acc=0.9788, F1=0.6803, Prec=0.7718, Rec=0.6633


In [18]:
import pandas as pd
import numpy as np

# Load the CSV
df = pd.read_csv("combined_dataset.csv")

images_without_labels = []

for idx, row in df.iterrows():
    # Convert label string to numpy array
    label_vector = np.array(list(map(float, row["labels"].split(","))))
    
    # Check if all zeros
    if np.all(label_vector == 0):
        images_without_labels.append(row["image_id"])

# Keep only unique image IDs
unique_images_without_labels = list(set(images_without_labels))

print(f"Total unique images with all-zero labels: {len(unique_images_without_labels)}")
if len(unique_images_without_labels) > 0:
    print("Unique image id's with all-zero labels:", unique_images_without_labels)

Total unique images with all-zero labels: 48
Unique image id's with all-zero labels: [198915, 42888, 542073, 58636, 382734, 41488, 536343, 550939, 101022, 127135, 228771, 344611, 261796, 458790, 308391, 267946, 447789, 402096, 260657, 270386, 370999, 330554, 176701, 477118, 226111, 64574, 121153, 404601, 49091, 268996, 98497, 320706, 374727, 476491, 528977, 556498, 200152, 461275, 310622, 240767, 312549, 514540, 560371, 278006, 25593, 481404, 502910, 173183]
